# Getting track ID's for my personal songs

In [ ]:
import requests
import pandas as pd

In [ ]:
mySongs =[]

In [ ]:
url = 'https://track-analysis.p.rapidapi.com/pktx/analysis'

querystring = {'song':'thinkingabtu','artist':'saraunh0ly'}

headers = {
}

response = requests.get(url, headers=headers, params=querystring)
mySongs.append(response.json())
print(mySongs)

[{'id': '8c060d0e68d661cb4e66e5b02d79b1e6', 'key': 'E', 'mode': 'major', 'camelot': '12B', 'tempo': 105, 'duration': '6:07', 'popularity': 54, 'energy': 70, 'danceability': 75, 'happiness': 34, 'acousticness': 85, 'instrumentalness': 89, 'liveness': 10, 'speechiness': 5, 'loudness': '-10 dB'}, {'id': 'f31699816bd21d591b62052fde690293', 'key': 'C#', 'mode': 'major', 'camelot': '3B', 'tempo': 156, 'duration': '2:22', 'popularity': 53, 'energy': 99, 'danceability': 51, 'happiness': 84, 'acousticness': 23, 'instrumentalness': 83, 'liveness': 11, 'speechiness': 8, 'loudness': '-5 dB'}, {'message': 'You have exceeded the DAILY quota for Requests on your current plan, BASIC. Upgrade your plan at https://rapidapi.com/soundnet-soundnet-default/api/track-analysis'}]


In [ ]:
personalDataset = pd.DataFrame(mySongs)
personalDataset.to_csv('/content/drive/MyDrive/personal_spotify_data.csv', index=False)

# Data Preprocessing

#### Getting our data and importing libraries

In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
filePath = '/content/drive/MyDrive/school classes/senior port/dataset.csv'
songDatasetog = pd.read_csv(filePath)
filePath2 = '/content/drive/MyDrive/school classes/senior port/personal_spotify_data.csv' 
personalDatasetog = pd.read_csv(filePath2)
## For my personal Data set.
# index 0 song: Magic Spells by Crystal Castles
# index 1 song: You Could Be The One by Snow Strippers
personalDataset = personalDatasetog
kaggleDataset = songDatasetog


#### Data info

In [53]:
kaggleDataset.info()
kaggleDataset.head(1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          11

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,...,-6.746,0,0.143,0.0322,0.000001,0.358,0.715,87.917,4,acoustic


In [ ]:
personalDataset.info()
personalDataset.head()

,id,key,mode,camelot,tempo,duration,popularity,energy,danceability,happiness,acousticness,instrumentalness,liveness,speechiness,loudness
0,8c060d0e68d661cb4e66e5b02d79b1e6,E,major,12B,105,6:07,54,70,75,34,85,89,10,5,-10 dB
1,f31699816bd21d591b62052fde690293,C#,major,3B,156,2:22,53,99,51,84,23,83,11,8,-5 dB


#### Cleaning methods and feature engineering

In [26]:
def cleanPersonalsongs(df: pd.DataFrame) -> pd.DataFrame:
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns = ['Unnamed: 0'])
    
    df = df.drop(columns = ['key', 'mode', 'camelot'])

    df.insert(1, 'artists', ['Crystal Castles', 'Snow Strippers'])
    df.insert(2, 'track_name', ['Magic Spells', 'You Could Be The One'])
    
    df = df.rename(columns={'happiness' : 'valence'})

    #Change the scale on my personal songs
    cols_to_divide = ['energy', 'danceability', 'valence', 'acousticness', 'instrumentalness', 'liveness', 'speechiness']
    for cols in cols_to_divide:
        df[cols] = df[cols] / 100
    
    df['loudness'] = df['loudness'].str.replace(' dB', '')
    df['loudness'] = df['loudness'].astype(float)

    #Engineer feature high score means a lot of energy and not a lot of happiness
    df.insert(3, 'energy_valence_ratio', df['energy'] / (df['valence'] + 0.001) )
    
    return df

In [27]:
def cleanKaggleSongs(df: pd.DataFrame) -> pd.DataFrame:
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

    df = df.dropna()

    df = df.drop(columns=['time_signature', 'mode', 'key'])

    #Engineer feature high score means a lot of energy and not a lot of happiness
    df.insert(5, 'energy_valence_ratio', df['energy'] / (df['valence'] + 0.001) )

    return df

#### Scaling our X

In [7]:
import numpy as np
from sklearn.preprocessing import StandardScaler

In [28]:
#Clean our data
dfPersonal = cleanPersonalsongs(personalDataset)
dfKaggle = cleanKaggleSongs(kaggleDataset)

In [38]:
scaler = StandardScaler()
features = ['energy_valence_ratio', 'tempo', 'popularity', 'energy', 'danceability', 'valence',
            'acousticness', 'instrumentalness', 'liveness', 'speechiness', 'loudness']
#finds the mean and standard deviation of columns then shifts the data so the mean is 0 and 1 Standard Deviation equals 1.0
xKaggle = scaler.fit_transform(dfKaggle[features])
xPersonal = scaler.transform(dfPersonal[features])

# Dimensionality Reduction

### PCA

In [10]:
from sklearn.decomposition import PCA

In [39]:
#Have to fix dim
pca = PCA(n_components=0.90)
#Finds the main axis where data is spread and compresses onto those axis
xKagglePca = pca.fit_transform(xKaggle)
xPersonalPca = pca.transform(xPersonal)

### Autoencoder

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim

In [40]:
class Autoencoder(nn.Module):

    def __init__(self, inputDim, latentDim):

        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(inputDim, 16),
            nn.ReLU(),
            nn.Linear(16, latentDim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latentDim, 16),
            nn.ReLU(),
            nn.Linear(16, inputDim)
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return latent, reconstructed

#### Variables

In [36]:
inputDim = xKaggle.shape[1]
latentDim = xKagglePca.shape[1]
model = Autoencoder(inputDim, latentDim)
crit = nn.MSELoss()
#pass the models weights into adam so he can update weights
opt = optim.Adam(model.parameters(), lr = 0.001)
tensorKaggle = torch.FloatTensor(xKaggle)

#### Training

In [ ]:
epochs = 50
#reset adam to zero, get recon, calc mse, backprop, adjust weights do again
for i in range(epochs):
    opt.zero_grad()
    _ , reconstruction = model(tensorKaggle)
    loss = crit(reconstruction, tensorKaggle)
    loss.backward()
    opt.step()

In [ ]:
#just get the latents without tracking gradients
with torch.no_grad():
    latentKaggleAE, _ = model(tensorKaggle)
    latentPerosnalAE, _ = model(torch.FloatTensor(xPersonal))

xKaggleAE = latentKaggleAE.numpy()
xPerosnalAE = latentPerosnalAE.numpy()

# JUNK